### 1. Imports & Data Loading

In [2]:
import os
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
def parse_netflix_fast(filepath, sample_every_n=5):
    """
    Reads a Netflix file fast using pandas.
    sample_every_n: keep 1 in every N users to reduce size.
    """
    rows = []
    current_movie = None
    
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.endswith(':'):
                current_movie = int(line[:-1])
            else:
                rows.append(line + f',{current_movie}')
    
    # Bulk parse all rows at once instead of one by one
    from io import StringIO
    raw = '\n'.join(rows)
    df = pd.read_csv(
        StringIO(raw),
        names=['user_id', 'rating', 'date', 'movie_id'],
        dtype={'user_id': np.int32, 'rating': np.int8, 'movie_id': np.int32}
    )
    df['date'] = pd.to_datetime(df['date'])
    
    # Sample users (keeps all ratings for sampled users, not random rows)
    if sample_every_n > 1:
        unique_users = df['user_id'].unique()
        kept_users = unique_users[::sample_every_n]  # every Nth user
        df = df[df['user_id'].isin(kept_users)]
    
    return df

# Load all 4 files
DATA_PATH = r'C:/Users/DELL/OneDrive/Desktop/Projects/Data science/netflix/netflix-recommender/data'

dfs = []
for i in range(1, 5):
    print(f"Loading file {i}...")
    df = parse_netflix_fast(os.path.join(DATA_PATH, f'combined_data_{i}.txt'), sample_every_n=10)
    dfs.append(df)
    print(f"  → {len(df):,} rows")

full_df = pd.concat(dfs, ignore_index=True)

# Still apply active user filter after
min_ratings = 20
active_users = full_df.groupby('user_id').size()
active_users = active_users[active_users >= min_ratings].index
full_df = full_df[full_df['user_id'].isin(active_users)]

print(f"\nFinal dataset: {len(full_df):,} ratings")
print(f"Users: {full_df['user_id'].nunique():,} | Movies: {full_df['movie_id'].nunique():,}")
print(f"Date range: {full_df['date'].min()} → {full_df['date'].max()}")

Loading file 1...
  → 2,406,128 rows
Loading file 2...
  → 2,694,523 rows
Loading file 3...
  → 2,253,873 rows
Loading file 4...
  → 2,669,534 rows

Final dataset: 9,407,204 ratings
Users: 96,650 | Movies: 17,765
Date range: 1999-12-09 00:00:00 → 2005-12-31 00:00:00


### 2. Train-Test Split (Time-based 80/20)

In [3]:
full_df = full_df.sort_values('date')

cutoff = full_df['date'].quantile(0.8)
print(f"Cutoff date: {cutoff}")

train_df = full_df[full_df['date'] <= cutoff]
test_df  = full_df[full_df['date'] >  cutoff]

# Only keep test rows for users and movies already in train
train_users  = set(train_df['user_id'])
train_movies = set(train_df['movie_id'])

test_df = test_df[
    test_df['user_id'].isin(train_users) &
    test_df['movie_id'].isin(train_movies)
]

print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")

Cutoff date: 2005-08-03 00:00:00
Train: 7,537,987 | Test: 1,086,209


In [ ]:
### 3. Build User-Item Matrix

In [4]:
# Create index maps (id → matrix index)
user_ids   = sorted(train_df['user_id'].unique())
movie_ids  = sorted(train_df['movie_id'].unique())

user_index_map  = {uid: i for i, uid in enumerate(user_ids)}
movie_index_map = {mid: i for i, mid in enumerate(movie_ids)}

rows = train_df['user_id'].map(user_index_map)
cols = train_df['movie_id'].map(movie_index_map)
vals = train_df['rating'].values

sparse_matrix = csr_matrix(
    (vals, (rows, cols)),
    shape=(len(user_ids), len(movie_ids))
)
train_matrix = sparse_matrix.toarray().astype(np.float32)

print(f"Matrix shape: {train_matrix.shape}")
print(f"Sparsity: {100 * (1 - np.count_nonzero(train_matrix) / train_matrix.size):.1f}%")

Matrix shape: (85337, 17168)
Sparsity: 99.5%


### 4. Evaluation Functions (MAP@10)

In [5]:
def average_precision_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    hits = 0
    score = 0.0
    for i, item in enumerate(recommended):
        if item in relevant:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(relevant), k) if relevant else 0.0

def map_at_k(recommendations_dict, test_data, k=10):
    """
    recommendations_dict: {user_id: [movie_id1, movie_id2, ...]}
    test_data: DataFrame with [user_id, movie_id, rating]
    """
    ap_scores = []
    for user_id, recommended in recommendations_dict.items():
        relevant = set(
            test_data[
                (test_data['user_id'] == user_id) &
                (test_data['rating'] >= 3.5)
            ]['movie_id']
        )
        if relevant:
            ap_scores.append(average_precision_at_k(recommended, relevant, k))
    return sum(ap_scores) / len(ap_scores) if ap_scores else 0.0

## 5. User-Based KNN Model

In [6]:
class UserBasedKNN:
    def __init__(self, k=20, batch_size=500):
        self.k = k
        self.batch_size = batch_size

    def fit(self, train_matrix):
        self.train_matrix = train_matrix
        self.sparse_matrix = csr_matrix(train_matrix)
        self.n_users = train_matrix.shape[0]
        print(f"Fitted on {self.n_users} users. Similarity computed on-demand.")

    def _get_user_similarities(self, user_idx):
        user_vec = self.sparse_matrix[user_idx]
        sims = cosine_similarity(user_vec, self.sparse_matrix).flatten()
        sims[user_idx] = 0
        return sims

    def predict_rating(self, user_idx, movie_idx):
        sim_scores = self._get_user_similarities(user_idx)
        rated_mask = self.train_matrix[:, movie_idx] > 0
        sim_scores = sim_scores * rated_mask
        top_k_users = np.argsort(sim_scores)[-self.k:]
        numerator   = np.sum(sim_scores[top_k_users] * self.train_matrix[top_k_users, movie_idx])
        denominator = np.sum(np.abs(sim_scores[top_k_users]))
        return numerator / denominator if denominator != 0 else 0

user_knn = UserBasedKNN(k=20)
user_knn.fit(train_matrix)

Fitted on 85337 users. Similarity computed on-demand.


### 6. Item-Based KNN Model

In [18]:
class ItemBasedKNN:
    def __init__(self, k=20):
        self.k = k

    def fit(self, train_matrix):
        self.train_matrix = train_matrix
        self.sparse_matrix = csr_matrix(train_matrix)
        self.n_items = train_matrix.shape[1]
        print(f"Fitted on {self.n_items} movies. Similarity computed on-demand.")

    def _get_item_similarities(self, movie_idx):
        item_vec = csr_matrix(self.sparse_matrix.T[movie_idx])  # force csr
        sims = cosine_similarity(item_vec, self.sparse_matrix.T).flatten()
        sims[movie_idx] = 0
        return sims

    def predict_rating(self, user_idx, movie_idx):
        sim_scores = np.asarray(self._get_item_similarities(movie_idx)).flatten()
        rated_mask = self.train_matrix[user_idx] > 0
        sim_scores = sim_scores * rated_mask
        top_k_items = np.argsort(sim_scores)[-self.k:]
        numerator   = np.sum(sim_scores[top_k_items] * self.train_matrix[user_idx, top_k_items])
        denominator = np.sum(np.abs(sim_scores[top_k_items]))
        return numerator / denominator if denominator != 0 else 0

# Refit
item_knn = ItemBasedKNN(k=20)
item_knn.fit(train_matrix)

# Quick test
test_pred = item_knn.predict_rating(0, 0)
print(f"Test pred: {test_pred:.3f}")

Fitted on 17168 movies. Similarity computed on-demand.
Test pred: 4.239


### 7. Candidate Generation & Recommendation Function

In [12]:
# Run this ONCE before the loop
movie_popularity = (train_matrix > 0).sum(axis=0)
top_candidate_movies = np.argsort(movie_popularity)[-200:]

def get_top_n_recommendations(model, user_idx, train_matrix, movie_index_map, n=10):
    idx_to_movie = {i: mid for mid, i in movie_index_map.items()}
    
    # Use pre-computed candidates
    unrated = [m for m in top_candidate_movies if train_matrix[user_idx, m] == 0]
    
    if hasattr(model, '_get_user_similarities'):
        sim_scores = np.asarray(model._get_user_similarities(user_idx)).flatten()
        preds = []
        for m in unrated:
            rated_mask = train_matrix[:, m] > 0
            s = sim_scores * rated_mask
            top_k = np.argsort(s)[-model.k:]
            num = np.sum(s[top_k] * train_matrix[top_k, m])
            den = np.sum(np.abs(s[top_k]))
            preds.append((idx_to_movie[m], num / den if den != 0 else 0))
    else:
        preds = []
        for m in unrated:
            sim_scores = np.asarray(model._get_item_similarities(m)).flatten()
            s = sim_scores * (train_matrix[user_idx] > 0)
            top_k = np.argsort(s)[-model.k:]
            num = np.sum(s[top_k] * train_matrix[user_idx, top_k])
            den = np.sum(np.abs(s[top_k]))
            preds.append((idx_to_movie[m], num / den if den != 0 else 0))
    
    preds.sort(key=lambda x: x[1], reverse=True)
    return [mid for mid, _ in preds[:n]]

# Time test
import time
uid = test_df['user_id'].unique()[0]
u_idx = user_index_map[uid]
start = time.time()
recs = get_top_n_recommendations(user_knn, u_idx, train_matrix, movie_index_map)
print(f"Time: {time.time()-start:.1f}s")
print(f"Recs: {recs}")

Time: 1.2s
Recs: [np.int32(14550), np.int32(14240), np.int32(11089), np.int32(7193), np.int32(12870), np.int32(11443), np.int32(14621), np.int32(12338), np.int32(16668), np.int32(11283)]


### 8. Generate Top-10 Recommendations (User-KNN)

In [15]:
test_users = test_df['user_id'].unique()[:500]

print("Generating User-KNN recommendations...")
user_knn_recs = {}
for i, uid in enumerate(test_users):
    if uid in user_index_map:
        u_idx = user_index_map[uid]
        user_knn_recs[uid] = get_top_n_recommendations(user_knn, u_idx, train_matrix, movie_index_map)
    if i % 50 == 0:
        print(f"  {i}/{len(test_users)} done...")


Generating User-KNN recommendations...
  0/500 done...
  50/500 done...
  100/500 done...
  150/500 done...
  200/500 done...
  250/500 done...
  300/500 done...
  350/500 done...
  400/500 done...
  450/500 done...


### 9. Generate Top-10 Recommendations (Item-KNN)

In [22]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("Pre-computing item similarity matrix for top 200 movies only...")
# Only compute similarity between top 200 candidate movies
top_200_vecs = item_knn.sparse_matrix.T[top_candidate_movies]  # shape: 200 x users
item_sim_matrix = cosine_similarity(top_200_vecs)  # shape: 200 x 200
np.fill_diagonal(item_sim_matrix, 0)
print(f"Item sim matrix shape: {item_sim_matrix.shape}")
print("Done! Now generating recommendations...")

# Fast version using pre-computed similarity
item_knn_recs = {}
item_test_users = test_df['user_id'].unique()[:200]
idx_to_movie = {i: mid for mid, i in movie_index_map.items()}

for i, uid in enumerate(item_test_users):
    if uid not in user_index_map:
        continue
    u_idx = user_index_map[uid]
    unrated = [j for j, m in enumerate(top_candidate_movies) if train_matrix[u_idx, m] == 0]
    
    preds = []
    for j in unrated:
        sim_scores = item_sim_matrix[j]
        rated_js = [jj for jj, m in enumerate(top_candidate_movies) if train_matrix[u_idx, m] > 0]
        if not rated_js:
            continue
        s = sim_scores[rated_js]
        top_k = np.argsort(s)[-20:]
        num = np.sum(s[top_k] * train_matrix[u_idx, top_candidate_movies[rated_js]][top_k])
        den = np.sum(np.abs(s[top_k]))
        preds.append((idx_to_movie[top_candidate_movies[j]], num / den if den != 0 else 0))
    
    preds.sort(key=lambda x: x[1], reverse=True)
    item_knn_recs[uid] = [mid for mid, _ in preds[:10]]
    
    if i % 50 == 0:
        print(f"  {i}/{len(item_test_users)} done...")

print("Done!")
print(f"Item-KNN: {len(item_knn_recs)} users")

Pre-computing item similarity matrix for top 200 movies only...
Item sim matrix shape: (200, 200)
Done! Now generating recommendations...
  0/200 done...
  50/200 done...
  100/200 done...
  150/200 done...
Done!
Item-KNN: 200 users


### 10. Evaluation — RMSE

In [25]:
def compute_rmse_fast(model, test_sample, user_index_map, movie_index_map, train_matrix):
    preds, actuals = [], []
    
    # Group by user so we compute similarities once per user
    for uid, group in test_sample.groupby('user_id'):
        u = user_index_map.get(uid)
        if u is None:
            continue
        
        # Compute user similarities once for all this user's test rows
        sim_scores = np.asarray(model._get_user_similarities(u)).flatten()
        
        for _, row in group.iterrows():
            m = movie_index_map.get(row['movie_id'])
            if m is None:
                continue
            rated_mask = train_matrix[:, m] > 0
            s = sim_scores * rated_mask
            top_k = np.argsort(s)[-model.k:]
            num = np.sum(s[top_k] * train_matrix[top_k, m])
            den = np.sum(np.abs(s[top_k]))
            if den > 0:
                preds.append(num / den)
                actuals.append(row['rating'])
    
    return np.sqrt(mean_squared_error(actuals, preds))

test_sample = test_df.sample(n=1000, random_state=42)

user_rmse = compute_rmse_fast(user_knn, test_sample, user_index_map, movie_index_map, train_matrix)
print(f"User-KNN RMSE: {user_rmse:.4f}")

item_rmse = compute_rmse(item_knn, test_sample, user_index_map, movie_index_map, train_matrix)
print(f"Item-KNN RMSE: {item_rmse:.4f}")

User-KNN RMSE: 0.9957
Item-KNN RMSE: 1.0240


### 11. Evaluation — MAP@10

In [26]:
user_map = map_at_k(user_knn_recs, test_df)
item_map = map_at_k(item_knn_recs, test_df)

print(f"User-KNN MAP@10: {user_map:.4f}")
print(f"Item-KNN MAP@10: {item_map:.4f}")

User-KNN MAP@10: 0.0217
Item-KNN MAP@10: 0.0111


### 12. Results Summary

In [27]:
results = pd.DataFrame({
    'Model':  ['User-KNN', 'Item-KNN', 'SVD (teammate)'],
    'RMSE':   [user_rmse, item_rmse, None],   # fill in SVD result
    'MAP@10': [user_map,  item_map,  None],   # fill in SVD result
})
print(results.to_string(index=False))

         Model     RMSE   MAP@10
      User-KNN 0.995713 0.021702
      Item-KNN 1.023951 0.011143
SVD (teammate)      NaN      NaN


### 13. Load Movie Titles

In [30]:
# Load movie titles
import os
movie_titles_path = os.path.join(DATA_PATH, 'movie_titles.csv')
movie_titles = pd.read_csv(movie_titles_path, encoding='latin-1', 
                           names=['movie_id', 'year', 'title'],
                           on_bad_lines='skip')
title_map = dict(zip(movie_titles['movie_id'], movie_titles['title']))
print(f"Loaded {len(title_map)} movie titles")
print(movie_titles.head())

def analyse_user_with_titles(uid, recs_dict, train_df, test_df):
    train_liked = train_df[(train_df['user_id'] == uid) & (train_df['rating'] >= 4)]['movie_id'].tolist()
    test_liked  = set(test_df[(test_df['user_id'] == uid) & (test_df['rating'] >= 3.5)]['movie_id'])
    recommended = recs_dict.get(uid, [])
    hits        = [m for m in recommended if m in test_liked]

    print(f"\nUser {uid}")
    print(f"  Liked in train:      {[title_map.get(m, m) for m in train_liked[:5]]}")
    print(f"  Recommended:         {[title_map.get(m, m) for m in recommended]}")
    print(f"  Hits in test:        {[title_map.get(m, m) for m in hits]}")
    print(f"  Precision@10:        {len(hits)/10:.2f}")

# Try several users to find a success case
for uid in list(user_knn_recs.keys())[:20]:
    test_liked = set(test_df[(test_df['user_id'] == uid) & (test_df['rating'] >= 3.5)]['movie_id'])
    recommended = user_knn_recs.get(uid, [])
    hits = [m for m in recommended if m in test_liked]
    if len(hits) > 0:
        print(f"Found success case: user {uid} with {len(hits)} hits")
        analyse_user_with_titles(uid, user_knn_recs, train_df, test_df)
        break

Loaded 17434 movie titles
   movie_id    year                         title
0         1  2003.0               Dinosaur Planet
1         2  2004.0    Isle of Man TT 2004 Review
2         3  1997.0                     Character
3         4  1994.0  Paula Abdul's Get Up & Dance
4         5  2004.0      The Rise and Fall of ECW
Found success case: user 698451 with 4 hits

User 698451
  Liked in train:      ['Star Wars: Episode VI: Return of the Jedi', 'Gladiator', 'The Lord of the Rings: The Fellowship of the Ring: Extended Edition', 'GoodFellas: Special Edition', 'Happy Gilmore']
  Recommended:         ['Lord of the Rings: The Fellowship of the Ring', 'Lord of the Rings: The Two Towers', 'The Godfather', 'Raiders of the Lost Ark', 'Saving Private Ryan', 'The Bourne Identity', 'Braveheart', 'Finding Nemo (Widescreen)', 'Indiana Jones and the Temple of Doom', 'Star Wars: Episode II: Attack of the Clones']
  Hits in test:        ['Lord of the Rings: The Two Towers', 'The Godfather', 'Saving 

### 14. Success & Failure Case Analysis

In [35]:

# FINAL ANALYSIS — SUCCESS & FAILURE CASES


# Success case
print("=== SUCCESS CASE ===")
for uid in list(user_knn_recs.keys())[:50]:
    test_liked = set(test_df[(test_df['user_id'] == uid) & (test_df['rating'] >= 3.5)]['movie_id'])
    hits = [m for m in user_knn_recs.get(uid, []) if m in test_liked]
    if len(hits) >= 3:
        analyse_user_with_titles(uid, user_knn_recs, train_df, test_df)
        break

# Failure case
print("\n=== FAILURE CASE ===")
for uid in list(user_knn_recs.keys()):
    train_count = len(train_df[train_df['user_id'] == uid])
    test_liked = set(test_df[(test_df['user_id'] == uid) & (test_df['rating'] >= 3.5)]['movie_id'])
    hits = [m for m in user_knn_recs.get(uid, []) if m in test_liked]
    if train_count < 25 and len(test_liked) > 0 and len(hits) == 0:
        print(f"(User has only {train_count} ratings in train — sparse user)")
        analyse_user_with_titles(uid, user_knn_recs, train_df, test_df)
        break

# Final results table
print("\n=== FINAL RESULTS ===")
results = pd.DataFrame({
    'Model':  ['User-KNN', 'Item-KNN', 'SVD (teammate)'],
    'RMSE':   [user_rmse, item_rmse, None],
    'MAP@10': [user_map,  item_map,  None]
})
print(results.to_string(index=False))

=== SUCCESS CASE ===

User 698451
  Liked in train:      ['Star Wars: Episode VI: Return of the Jedi', 'Gladiator', 'The Lord of the Rings: The Fellowship of the Ring: Extended Edition', 'GoodFellas: Special Edition', 'Happy Gilmore']
  Recommended:         ['Lord of the Rings: The Fellowship of the Ring', 'Lord of the Rings: The Two Towers', 'The Godfather', 'Raiders of the Lost Ark', 'Saving Private Ryan', 'The Bourne Identity', 'Braveheart', 'Finding Nemo (Widescreen)', 'Indiana Jones and the Temple of Doom', 'Star Wars: Episode II: Attack of the Clones']
  Hits in test:        ['Lord of the Rings: The Two Towers', 'The Godfather', 'Saving Private Ryan', 'Star Wars: Episode II: Attack of the Clones']
  Precision@10:        0.40

=== FAILURE CASE ===
(User has only 1 ratings in train — sparse user)

User 1356051
  Liked in train:      ['XXX: State of the Union']
  Recommended:         ['The Green Mile', 'Gladiator', 'X-Men', 'Shrek (Full-screen)', 'The Incredibles', 'Indiana Jones an